# 02. Modeling & Evaluation Pipeline
## Customer Sentiment Analysis & Reporting for E-commerce

This notebook walks through the machine learning pipeline:
1. Custom text preprocessing (tokenization, stop words preserving negations, lemmatization)
2. Feature representation comparison: Bag-of-Words vs. TF-IDF
3. Training baseline (MultinomialNB) and candidate models (Logistic Regression, Random Forest, Gradient Boosting)
4. Hyperparameter tuning using stratified 5-fold cross-validation
5. Final evaluation on held-out test split (macro-F1 >= 0.65)
6. Error analysis and ethical bias evaluation across departments and age groups

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline

from src.models.train import load_dataset_from_db_or_clean
from src.features.text import TextPreprocessor, build_bow_vectorizer, build_tfidf_vectorizer
from src.models.predict import ModelService
from src.models.evaluate import compute_metrics, run_error_analysis, run_ethics_evaluation

### 1. Load Stratified Splits
Ensuring strict isolation: train (70%), validation (15%), test (15%).

In [ ]:
df = load_dataset_from_db_or_clean()
train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']
test_df = df[df['split'] == 'test']

print(f'Train split: {len(train_df)} rows')
print(f'Validation split: {len(val_df)} rows')
print(f'Test split: {len(test_df)} rows')

### 2. Inspect Active Model Pipeline & Artifacts

In [ ]:
service = ModelService()
print(f'Active model version: {service.version}')
print('Pipeline structure:', service.pipeline)

### 3. Final Test Set Evaluation
Evaluated on held-out test set once with complete metrics and confusion matrix.

In [ ]:
y_test_pred = service.pipeline.predict(test_df['review_text'])
metrics = compute_metrics(test_df['true_label'], y_test_pred)

print(f'Test Accuracy:  {metrics["accuracy"]:.4f}')
print(f'Test Macro-F1:  {metrics["macro_f1"]:.4f} (Requirement >= 0.65)')
print(f'Test Weighted-F1: {metrics["weighted_f1"]:.4f}')
print('\nPer-class metrics:')
for label, scores in metrics['per_class'].items():
    print(f'  {label:8s} -> Precision: {scores["precision"]:.4f}, Recall: {scores["recall"]:.4f}, F1: {scores["f1"]:.4f}')

### 4. Ethical Bias & Demographic Evaluation
Inspecting performance across product departments and customer age bands (<30, 30-45, >45).

In [ ]:
ethics_results = run_ethics_evaluation(service.pipeline, test_df)
print('Department Breakdown:')
for dept, res in ethics_results['by_department'].items():
    print(f'  {dept:15s} (n={res["count"]:4d}) -> Accuracy: {res["accuracy"]:.4f}, Macro-F1: {res["macro_f1"]:.4f}')

print('\nAge Group Breakdown:')
for group, res in ethics_results['by_age_group'].items():
    print(f'  {group:15s} (n={res["count"]:4d}) -> Accuracy: {res["accuracy"]:.4f}, Macro-F1: {res["macro_f1"]:.4f}')

### 5. Interactive Prediction Verification

In [ ]:
sample_texts = [
    'This dress fits like a glove and the silk fabric is breathtaking!',
    'It is okay, but runs slightly small and the color is duller than pictured.',
    'Terrible quality! Shrunk in cold wash and seams ripped on day one. Do not buy.'
]
results = service.predict_batch(sample_texts, persist=False)
for text, res in zip(sample_texts, results):
    print(f'Review: "{text}"')
    print(f'  -> Predicted: {res["predicted_label"]} (confidence: {res["confidence"]:.3f})')
    print(f'  -> Probabilities: {res["probabilities"]}\n')